# LLM으로 오기까지 ①: 고정 길이 시대에서 어텐션까지 (1회차)

오늘 쓰는 임베딩 모델, 번역 모델, LLM은 하루아침에 나온 것이 아니다. 앞 세대 방법이 부딪힌 **문제**를 다음 세대가 **해결**하면서 왔고, 해결책은 늘 **새 제약**을 남겼다.
이 노트북은 그 흐름을 셀 하나당 1~3분 안에 직접 확인하도록 구성했다.
1회차 명제(모든 AI 응용은 분류 또는 회귀로 귀결된다)로 보면, 각 절의 모델이 마지막에 하는 일(다음 단어·후보 고르기)은 같고 **입력 표현**이 바뀌어 온 것이다.

| 절 | 문제 | 해결 | 확인 방법 | 시간 |
|---|---|---|---|---|
| 1 | 본 적 없는 단어 조합은 확률 0 (n-gram 희소성) | 단어를 벡터로(신경망 언어모델) | bigram 언어모델 | 3분 |
| 2 | 문장을 고정 길이 벡터로 만들면 어순이 사라짐 | 순서를 읽는 모델(RNN → 인코더) | 단어 주머니 비교 | 2분 |
| 3 | 키워드가 안 겹치면 못 찾음, 단어 하나에 뜻 하나 | 문장 의미 임베딩(Encoder) | TF-IDF vs 임베딩 | 5분 |
| 4 | 길이가 다른 문장으로 바꾸기(번역) | 인코더-디코더 = 조건부 언어모델 P(y│c) | 번역 모델 실행 | 3분 |
| 5 | 문장 전체를 벡터 하나(c)에 우겨넣는 병목 | 어텐션: 매 시점 입력 전체를 다시 봄 | 세 구조 직접 학습 | 15분 |
| 6 | 같은 부품, 다른 목적 | 구조 분기(Encoder / Enc-Dec / Decoder) | 구조별 비교표 | 5분 |

각 절은 **문제 → 확인 → 해결 → 확인 → 실제 응용 → 이후 회차 연결** 순서로 되어 있다. 위에서부터 차례로 실행한다.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src") if os.path.isdir("../src") else os.path.abspath("src"))
import config, utils, lineage
utils.korean_font()
print("도메인:", config.DOMAIN, "| 임베딩:", config.EMBED_MODEL)

## 1. 확률 언어모델과 n-gram 희소성 (~'13)

**문제.** 초기 언어모델은 "앞 단어 다음에 이 단어가 올 확률"을 학습 데이터에서 **세어서** 구했다(n-gram).
그러면 데이터에 한 번도 안 나온 조합은 확률이 0이 된다. 말이 되는 문장인데도 "불가능한 문장"으로 판정된다.

아래는 게임 도메인 문서로 만든 bigram(앞 1단어) 언어모델이다. 두 번째·세 번째 문장은 말이 되지만 확률이 0이 된다.

In [ ]:
sents = lineage.domain_sentences()
uni, bi = lineage.bigram_lm(sents)
tests = ["정기 점검은 매주 수요일 06:00~08:00에 진행한다",
         "정기 점검은 매주 목요일에 진행한다",
         "점검 보상으로 루멘석 100개를 지급한다"]
utils.print_table([lineage.sentence_prob(s, uni, bi) for s in tests]);

**임시 처방: 스무딩.** 모든 조합에 작은 값을 더해 0을 피한다(add-one). 0은 사라지지만 "목요일"이 "수요일"과 비슷한 단어라는 지식은 여전히 없다.

In [ ]:
utils.print_table([{**lineage.sentence_prob(s, uni, bi, smoothing=1.0), "스무딩": "add-one"} for s in tests]);

**해결의 방향.** 신경망 언어모델(Bengio et al., 2003)은 단어를 **연속 벡터**로 바꿔서 "수요일·목요일처럼 비슷한 단어는 비슷한 벡터"가 되게 했다. 본 적 없는 조합도 비슷한 단어의 통계를 빌려 확률을 줄 수 있다. 이것이 오늘 쓰는 임베딩의 출발점이다.

- **실제 응용(당시):** 검색어 자동완성, 맞춤법 교정, 통계적 기계번역(SMT)의 언어모델 부품
- **남긴 제약:** 직전 n개 단어만 보는 **고정 윈도우**. 문장 앞쪽 정보는 버린다

## 2. 고정 길이 벡터와 어순 소실

**문제.** 신경망(MLP 등)은 입력 크기가 고정되어야 한다. 길이가 제각각인 문장을 넣으려고 **단어 주머니(BoW)**(단어별 개수를 센 고정 길이 벡터)로 바꿔 썼다.
그런데 이렇게 하면 **어순이 사라진다.** 고객센터라면 치명적인 예다.

In [ ]:
a, b = "환불은 되는데 교환은 안 돼요", "교환은 되는데 환불은 안 돼요"
print(a, "→", lineage.bow(a))
print(b, "→", lineage.bow(b))
print("두 벡터가 같은가?", lineage.bow(a) == lineage.bow(b))

**해결의 방향.** 순서대로 한 단어씩 읽으며 상태를 갱신하는 **RNN**(LSTM '97, GRU '14)이 가변 길이 입력을 직접 다루게 했다. 이것이 인코더의 원형이다.

- **실제 응용:** 감성 분석·스팸 분류는 BoW + 분류기로도 꽤 됐다(출력이 라벨 하나라서). 어순이 중요한 번역·대화에서 한계가 드러났다
- **남긴 제약:** RNN은 출력도 입력과 같은 길이(품사 태깅)거나 라벨 하나(분류)일 때만 쉽다 → 길이가 다른 문장으로 **바꾸는** 문제는 4절

## 3. 키워드 겹침에서 의미로: 문장 임베딩

**문제 ①: 키워드가 안 겹치면 못 찾는다.** 글자 TF-IDF(단어 주머니의 개선판)로 FAQ를 검색하면 표현을 바꾼 질문을 놓친다.

**문제 ②: 단어 하나에 벡터 하나.** word2vec('13) 같은 단어 임베딩은 "배"가 어느 문장에 있든 같은 벡터다. 다의어(배: 신체·선박·과일)를 구분하지 못한다.

In [ ]:
utils.print_table(lineage.static_token_ids("배", ["배가 아파서 병원에 갔다", "배를 타고 섬에 갔다", "과일 가게에서 배를 샀다"]))
print("→ 토큰 ID가 같다 = 입력 임베딩 표에서 같은 행(같은 벡터)을 꺼낸다")

**해결: 문장 전체를 보고 만든 의미 벡터(Encoder).** BERT 계열 인코더는 문장 전체를 양방향으로 읽어 문맥에 따라 다른 벡터를 만든다. 문장 임베딩 모델(Sentence-BERT '19 계열, 오늘 쓰는 KURE)은 그 결과를 문장 하나당 벡터 하나로 만든다.

아래 표에서 키워드 방식과 의미 방식이 고르는 1등이 어떻게 다른지 본다.

In [ ]:
import model_utils
emb = model_utils.load_embedder()
utils.print_table(lineage.keyword_vs_meaning("배가 아파서 아무것도 못 먹겠어요",
    ["복통이 심해서 병원에 가야 할 것 같아요", "배를 타고 제주도에 갔어요", "과일 가게에서 배를 샀어요", "속이 쓰리고 소화가 안 돼요"], emb));

이제 FAQ 검색기 전체로 두 방식을 비교한다(실습 1의 본 과제). 같은 질의 10건, 같은 FAQ 25건이다.

In [ ]:
from faq_search import FaqSearcher, evaluate
faq = utils.read_json(config.data_dir() / "faq.json")
queries = utils.read_json(config.data_dir() / "faq_queries.json")
for method in ("tfidf", "embed"):
    s = FaqSearcher(faq, method, embedder=emb)   # 위에서 불러온 임베딩 모델 재사용
    r = evaluate(s, queries, None)
    print(f"[{method}] 1순위 적중 {r['hits']}/{r['n']}")
    utils.print_table(r["rows"])

**남긴 제약: 직접 확인.** 문장 임베딩도 만능이 아니다. 아래 두 문장은 뜻이 정반대인데 의미 벡터가 거의 같다. 주제가 비슷하면 가깝게 나오고, **누가 무엇을 부정했는지** 같은 세밀한 논리는 놓친다.

In [ ]:
utils.print_table(lineage.keyword_vs_meaning("환불은 되는데 교환은 안 돼요",
    ["교환은 되는데 환불은 안 돼요", "환불 가능하지만 교환은 불가합니다"], emb));

- **실제 응용:** 검색 엔진의 의미 검색, 고객센터 FAQ 매칭, 중복 문의 묶기, 추천
- **이후 회차:** 이 Encoder가 **6회차 RAG의 검색기**가 된다. 위의 세밀한 논리 문제는 6회차 재순위화(Cross-Encoder)와 LLM 생성 단계가 맡는다
- **과제 카드 B:** `python src/faq_search.py --threshold 0.5`. 기준 미달이면 "해당 없음". 기준을 올리고 내리면 놓치는 질문과 잘못 찾는 질문이 반대로 움직인다

## 4. 인코더-디코더: 디코더는 조건부 언어모델이다 ('13~'14)

**문제.** 번역처럼 입력과 출력의 **길이가 다르고 단어 대응(정렬)이 주어지지 않는** 변환은 RNN 하나로 풀기 어려웠다. 당시에는 여러 모듈을 따로 학습해 붙인 통계적 기계번역(SMT)이 담당했고, 모듈마다 오류가 쌓였다.

**해결.** 인코더가 입력을 읽어 문맥 c를 만들고, 디코더가 c를 조건으로 한 토큰씩 생성한다.

- 언어모델: P(y) = Π P(y_t │ y_<t)
- 인코더-디코더: P(y │ x) = Π P(y_t │ y_<t, **c**)  ← 디코더 = 조건 c가 붙은 언어모델

이 "조건 c를 무엇으로 채우는가"가 과정 전체에 걸친 질문이다(5회차 프롬프트, 6회차 검색 결과).

In [ ]:
from seq2seq_demo import run_encdec
row = run_encdec(["결제했는데 아이템이 안 들어왔어요.", "지난 패치 이후 서리칼날의 공격력이 120에서 105로 낮아졌습니다."])
print(row["입력→출력"])

- **실제 응용:** 2016년 신경망 번역 상용화(SMT 대체), 요약, 대화 생성, 음성 인식
- **남긴 제약:** 문장이 길어도 c는 **고정 길이 벡터 하나**다 → 5절에서 직접 확인

## 5. 고정 길이 벡터 병목 → 어텐션 → 트랜스포머 (실습 3 핵심)

같은 과제(한글 수사 → 숫자: "삼만 오천이백 원" → "35200원"), 같은 데이터, 같은 학습 시간으로 세 구조를 학습한다.

| 구조 | 디코더가 보는 것 | 연도 |
|---|---|---|
| `gru` | 인코더 마지막 상태 c **하나** | Cho·Sutskever 2014 |
| `gru_attn` | 매 시점 인코더 **전체 상태의 가중합** (어텐션) | Bahdanau 2014 |
| `transformer` | RNN 없이 어텐션만 | Vaswani 2017 |

터미널에서 돌리는 편이 편하다(셋을 차례로, 구조당 2분):
```
python src/tiny_seq2seq_train.py --arch all --minutes 2
```
노트북에서 돌리려면 아래 셀의 주석을 푼다.

In [ ]:
# import subprocess; subprocess.run([sys.executable, "../src/tiny_seq2seq_train.py", "--arch", "all", "--minutes", "2"])
rows = []
for arch in ("gru", "gru_attn", "transformer"):
    p = config.PATHS["results"] / f"numerals_{arch}.json"
    if p.exists():
        r = utils.read_json(p)
        rows.append({"구조": arch, "파라미터": r["params"], "검증 정확도": r["valid_acc"], **{f"{k}자": v for k, v in r["valid_by_len"].items()},
                     "긴 입력(16자+)": r["long_acc"], "초당 학습 문장": r["sent_per_sec"]})
utils.print_table(rows);

In [ ]:
from IPython.display import Image, display
for f in ("numerals_by_length.png", "numerals_loss.png"):
    if (config.PATHS["figs"] / f).exists():
        display(Image(str(config.PATHS["figs"] / f)))

**읽는 법.**
- `gru`: 짧은 입력은 잘 맞히는데 **입력이 길어질수록 정확도가 떨어진다.** 긴 금액을 벡터 하나에 다 담지 못한 것이다(고정 길이 병목)
- `gru_attn`: 길이와 상관없이 맞힌다. 디코더가 자릿수마다 입력의 해당 부분을 다시 보기 때문이다
- 병목을 더 좁혀 보기: `python src/tiny_seq2seq_train.py --arch gru --hidden 32`

**어텐션이 실제로 무엇을 보는지** 확인한다. 출력 숫자 하나하나가 입력의 어느 글자를 봤는지가 대각선 모양으로 드러난다(정렬을 학습으로 익힘).

In [ ]:
import subprocess
for arch in ("gru_attn", "transformer"):
    if (config.PATHS["runs"] / f"numerals_{arch}.pt").exists():
        subprocess.run([sys.executable, os.path.join(os.path.dirname(config.__file__), "viz_attention.py"), "tiny", "--arch", arch, "--text", "삼억 칠천이백만 오천 원"])
        display(Image(str(config.PATHS["figs"] / f"attn_tiny_{arch}.png")))

- **실제 응용:** 어텐션 도입 후 번역 품질이 크게 오르고 정렬을 사람이 만들 필요가 없어졌다. 음성 인식·이미지 캡션에도 같은 원리가 쓰였다
- **남긴 제약:** RNN은 한 시점씩 **차례로** 계산해야 해서 GPU로 병렬화하기 어렵다(위 표의 학습 속도 비교, 코드의 `for t in range(...)` 반복문)
- **이어서:** 트랜스포머는 RNN을 **빼고** 어텐션만 남겼다. 그래서 학습을 한 번에 병렬로 돌릴 수 있게 됐다 → 2회차에서 내부를 연다

> 오개념 정정: 어텐션을 처음 도입한 것은 트랜스포머가 아니다(2014년 RNN 인코더-디코더에 먼저 들어감). 트랜스포머의 기여는 **RNN 제거**다.

## 6. 같은 부품, 다른 목적: 구조 분기

세 구조를 같은 문장으로 돌려 **구조별 모델 비교표**(오늘 산출물)를 만든다.
```
python src/seq2seq_demo.py
```

In [ ]:
p = config.PATHS["results"] / "structure_compare.md"
print(p.read_text(encoding="utf-8") if p.exists() else "터미널에서 python src/seq2seq_demo.py를 먼저 실행")

| 구조 | 사전학습 목표 | 주 용도 | 대표 모델 |
|---|---|---|---|
| 인코더 전용 | 빈칸 맞히기(양방향) | 분류, 임베딩·검색 | BERT, KURE |
| 인코더-디코더 | 잡음 제거·범위 복원 | 번역, 요약, 음성 인식 | T5, BART, Whisper, opus-mt |
| 디코더 전용 | 다음 토큰 예측(단방향) | 생성, 범용 LLM | GPT, Qwen, A.X |

**분기는 비용 때문이 아니라 목적(사전학습 목표) 때문이고, 디코더 전용으로의 수렴은 규모(스케일) 때문이다.** 2회차에서 이어서 본다.

> 오개념 정정: 감성 분석은 문장 → 라벨 하나라서 디코더가 필요 없다. 인코더-디코더가 퍼진 곳은 출력이 문장인 번역·요약·대화다.

## 정리: 오늘 확인한 "문제 → 해결 → 남긴 제약"

| 단계 | 해결한 문제 | 남긴 제약 | 오늘 확인한 값(직접 기입) |
|---|---|---|---|
| n-gram → 신경망 언어모델 | 본 적 없는 조합 = 확률 0 | 고정 윈도우 | 목요일 문장 log P = |
| BoW → RNN | 가변 길이 입력 | 출력 형태 제한 | 어순 바꾼 두 문장 벡터 같음? |
| 키워드 → 문장 임베딩 | 표현이 달라도 찾음 | 세밀한 논리 놓침 | TF-IDF __/10, 임베딩 __/10 |
| 인코더-디코더 | 길이가 다른 변환 | 고정 길이 벡터 c 병목 | gru 긴 입력 정확도 = |
| 어텐션 | 병목 해소, 정렬 자동 학습 | 순차 계산 → 병렬화 불가 | gru_attn 긴 입력 정확도 = |
| 트랜스포머 | 병렬화 | 길이 제곱 비용, 기억 없음 | (2회차) |

**세 가지 관통 축**: 이후 모든 회차에서 다시 나온다.
1. **고정 길이 제약의 이동:** 입출력 길이 → 문맥 벡터 c → 컨텍스트 윈도우 → 검색·메모리로 외부화
2. **조건부 생성 P(y│c):** 발전사는 "조건 c를 무엇으로, 누가 채우는가"의 변화다
3. **지식의 저장 위치:** 가중치 안(학습) vs 바깥 저장소(검색)